# S3ANet Automated Experiments
This notebook runs the S3ANet model on all datasets (PaviaU, Salinas, Houston, IndianP), stores the results in proper subfolders, and generates a final Excel report.


In [ ]:
# Install required packages
!pip install openpyxl


## 1. Mount Google Drive and Setup Environment
Assuming your dataset `.mat` files are stored in your Google Drive under a folder named `SACNEet_Data`.
Please modify the `drive_datasets_path` below if they are stored in a different folder.


In [ ]:
from google.colab import drive
import os
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Ensure the S3ANet project is your current working directory
# Note: In Colab, you might need to clone your repo first if not already in it.
import os
if not os.path.exists('/content/S3ANET'):
    !git clone https://github.com/SRUJANPATEL3669/S3ANET.git
%cd /content/S3ANET

# Create Data folder
os.makedirs('./Data', exist_ok=True)

# Path to the datasets in your Google Drive (Modify if needed)
drive_datasets_path = '/content/drive/MyDrive/SACNet_Data'

if os.path.exists(drive_datasets_path):
    print("Copying datasets from Google Drive...")
    for file in os.listdir(drive_datasets_path):
        if file.endswith('.mat'):
            shutil.copy(os.path.join(drive_datasets_path, file), './Data/')
    print("Datasets copied successfully!")
else:
    print(f"Warning: The path {drive_datasets_path} does not exist in your Google Drive.")
    print("Please make sure the datasets are present in that folder.")


## 2. Generate Data Samples (`.npy` files)
This step runs `GenSample.py` for datasets 1 to 4 to prepare the training/testing arrays.


In [ ]:
# Run GenSample.py for datasets 1 to 4
for data_id in range(1, 5):
    print(f"\nGenerating samples for Data ID {data_id}...")
    !python GenSample.py --dataID {data_id}
print("\nData generation complete.")


## 3. Run Experiments and Collect Results
This cell will execute the `Attack_FGSM_S3ANet.py` script for all datasets.
It will capture the console output to extract Accuracy metrics (OA, Kappa, AA, per-class accuracies) and store them in a Pandas DataFrame.
Finally, the DataFrame will be exported to an Excel file (`Final_Results.xlsx`).


In [ ]:
# Run GenAdvExample.py (Adversarial Example Generation)
# Results will be saved directly to Google Drive
drive_results_path = '/content/drive/MyDrive/S3ANet_Results/'
import os
os.makedirs(drive_results_path, exist_ok=True)

for data_id in range(1, 5):
    print(f'\nGenerating adversarial examples for Data ID {data_id}...')
    !python GenAdvExample.py --dataID {data_id} --save_path_prefix {drive_results_path}
print('\nAdversarial example generation complete.')


In [ ]:
import subprocess
import pandas as pd
import re

datasets = {1: 'PaviaU', 2: 'Salinas', 3: 'Houston', 4: 'IndianP'}
all_results = []

for data_id, dataset_name in datasets.items():
    print(f"\n================ Running Experiment for {dataset_name} ================")
    
    # Define the save prefix so results go to a specific folder
    # Define the save prefix so results go to Google Drive
    save_prefix = f"/content/drive/MyDrive/S3ANet_Results/Attack_Results/{dataset_name}/"
    os.makedirs(save_prefix, exist_ok=True)
    
    # Run the script and capture output
    command = f"python Attack_FGSM_S3ANet.py --dataID {data_id} --save_path_prefix {save_prefix}"
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    
    output_lines = []
    for line in process.stdout:
        print(line, end='')  # Print out the logs in real-time
        output_lines.append(line.strip())
        
    process.wait()
    
    # Parse the final metrics from the output
    oa, kappa, aa, train_time, test_time, runtime = None, None, None, None, None, None
    producer_a = ""
    
    for i, line in enumerate(output_lines):
        if "OA=" in line and "Kappa=" in line:
            match = re.search(r'OA=([\d\.]+),Kappa=([\d\.]+)', line)
            if match:
                oa = float(match.group(1))
                kappa = float(match.group(2))
        
        if "producerA:" in line:
            # Join the array string that might span multiple lines
            prod_str = line.split("producerA:")[1].strip()
            producer_a = prod_str
        
        if "AA=" in line:
            match = re.search(r'AA=([\d\.]+)', line)
            if match:
                aa = float(match.group(1))
                
        if "Train_time:" in line and "Test_time:" in line:
            match = re.search(r'Train_time: ([\d\.]+), Test_time: ([\d\.]+), Runtime: ([\d\.]+)', line)
            if match:
                train_time = float(match.group(1))
                test_time = float(match.group(2))
                runtime = float(match.group(3))
                
    # Append to results
    all_results.append({
        'Dataset': dataset_name,
        'OA (%)': oa,
        'Kappa (%)': kappa,
        'AA (%)': aa,
        'Producer A': producer_a,
        'Train Time (s)': train_time,
        'Test Time (s)': test_time,
        'Total Runtime (s)': runtime
    })

print("\nAll experiments finished. Generating Excel report...")

# Create DataFrame and save to Excel
df = pd.DataFrame(all_results)
excel_path = '/content/drive/MyDrive/S3ANet_Results/Final_Results_3D.xlsx'
df.to_excel(excel_path, index=False)
print(f"Results successfully saved to {excel_path}!")
df  # Display the dataframe in Colab
